## Data Loading and Packages

In [ ]:
### Load libraries
import pandas as pd
import numpy as np
import scipy.sparse
from sklearn.decomposition import PCA

## Load data
cols_used = ["player_name",
    "pitcher",
    "pitch_type",
    "release_speed",
    "release_spin_rate",
    "spin_axis",
    "release_pos_x",
    "release_pos_y",
    "release_pos_z",
    "release_extension",
    "vx0",
    "vy0",
    "vz0",
    "ax",
    "ay",
    "az"]

# game_year lets us localize pitcher arsenal comparisons to the season a pitch was thrown
load_cols = cols_used + ["game_year"]

# usecols avoids reading all ~119 raw Statcast columns before subsetting
data = pd.read_csv(
    r'C:\Users\choul\OneDrive\Baseball Repo\Baseball-Analytics\data\MLB_2021-2025.csv',
    usecols=load_cols
).dropna(subset=load_cols)

data.head()

## Feature Engineering

#### Trajectory Calculation

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

TRAJ_DISTANCES = [10, 20, 30, 40]

PITCHER_COL = "pitcher"
PITCH_TYPE_COL = "pitch_type"
SEASON_COL = "game_year"

# Non-standard / misc pitch codes out of scope for a Stuff+ model
# (pitchouts, unknowns, eephus, forkball, knuckleball, slow curve, screwball, generic "fastball")
JUNK_PITCH_TYPES = {"FA", "EP", "FO", "KN", "CS", "SC", "PO", "UN"}


# ============================================================
# V1: PHYSICAL PITCH REPRESENTATION
# ============================================================

PHYSICS_COLS = [
    "release_speed",
    "release_spin_rate",
    "spin_axis",
    "release_pos_x",
    "release_pos_y",
    "release_pos_z",
    "release_extension",
    "vx0",
    "vy0",
    "vz0",
    "ax",
    "ay",
    "az",
]

required_cols = [PITCHER_COL, PITCH_TYPE_COL, SEASON_COL] + PHYSICS_COLS

df = data.copy()

# Drop pitch types out of scope for Stuff+ before anything else -- this is a
# modeling-scope decision, distinct from the NaN-based data-quality dropna below.
df = df[~df[PITCH_TYPE_COL].isin(JUNK_PITCH_TYPES)].copy()

# Only keep pitches with complete physics data
df = df.dropna(subset=required_cols).copy()


# ============================================================
# SPIN AXIS
# ============================================================

# Spin axis is circular, so represent it as sin/cos rather
# than treating degrees as a normal continuous variable.

theta = np.deg2rad(df["spin_axis"])

df["spin_axis_sin"] = np.sin(theta)
df["spin_axis_cos"] = np.cos(theta)


# ============================================================
# BASIC DERIVED PHYSICS
# ============================================================

df["velocity_mag"] = np.sqrt(
    df["vx0"]**2 +
    df["vy0"]**2 +
    df["vz0"]**2
)

df["acceleration_mag"] = np.sqrt(
    df["ax"]**2 +
    df["ay"]**2 +
    df["az"]**2
)

df["horizontal_velocity"] = np.sqrt(
    df["vx0"]**2 +
    df["vy0"]**2
)

df["horizontal_acceleration"] = np.sqrt(
    df["ax"]**2 +
    df["ay"]**2
)


# ============================================================
# TRAJECTORY FUNCTIONS (VECTORIZED)
# ============================================================

def solve_time_to_y_vectorized(y0, vy0, ay, distance):
    """
    Vectorized solve for the smallest positive time at which a pitch
    (starting at y0, with velocity vy0 and acceleration ay) reaches
    `distance` feet from the release point:

        target_y = y0 - distance
        target_y = y0 + vy0*t + 0.5*ay*t^2

    Since target_y = y0 - distance, the "c" term of the quadratic
    (y0 - target_y) simplifies to `distance` directly.

    Branch selection (linear vs. quadratic) and the discriminant<0
    gate mirror the original scalar solve_time_to_y exactly, just
    applied elementwise via boolean masks instead of Python if/else.
    """

    a = 0.5 * ay
    b = vy0
    c = float(distance)

    with np.errstate(invalid="ignore", divide="ignore"):
        discriminant = b**2 - 4 * a * c
        valid_disc = discriminant >= 0
        sqrt_disc = np.sqrt(np.clip(discriminant, 0, None))

        is_linear = np.abs(a) < 1e-10
        b_nonzero = np.abs(b) >= 1e-10

        t_linear = np.where(b_nonzero, -c / np.where(b_nonzero, b, 1.0), np.nan)
        t_linear = np.where(t_linear > 0, t_linear, np.nan)

        safe_a = np.where(is_linear, 1.0, a)
        t1 = (-b + sqrt_disc) / (2 * safe_a)
        t2 = (-b - sqrt_disc) / (2 * safe_a)
        t1 = np.where(t1 > 0, t1, np.inf)
        t2 = np.where(t2 > 0, t2, np.inf)
        t_quad = np.minimum(t1, t2)
        t_quad = np.where(np.isinf(t_quad), np.nan, t_quad)

        t = np.where(is_linear, t_linear, t_quad)
        t = np.where(valid_disc, t, np.nan)

    return t


# ============================================================
# CREATE V1 TRAJECTORY FEATURES
# ============================================================

for distance in TRAJ_DISTANCES:

    y0 = df["release_pos_y"].to_numpy()
    vy0 = df["vy0"].to_numpy()
    ay = df["ay"].to_numpy()
    x0 = df["release_pos_x"].to_numpy()
    vx0 = df["vx0"].to_numpy()
    ax = df["ax"].to_numpy()
    z0 = df["release_pos_z"].to_numpy()
    vz0 = df["vz0"].to_numpy()
    az = df["az"].to_numpy()

    t = solve_time_to_y_vectorized(y0, vy0, ay, distance)

    # NaN in t propagates naturally through this arithmetic, matching the
    # original's "return all-NaN if t is NaN" behavior with no extra masking.
    with np.errstate(invalid="ignore"):
        x = x0 + vx0 * t + 0.5 * ax * t**2
        z = z0 + vz0 * t + 0.5 * az * t**2
        vx = vx0 + ax * t
        vy = vy0 + ay * t
        vz = vz0 + az * t
        speed = np.sqrt(vx**2 + vy**2 + vz**2)

    df[f"x_{distance}ft"] = x
    df[f"z_{distance}ft"] = z
    df[f"vx_{distance}ft"] = vx
    df[f"vy_{distance}ft"] = vy
    df[f"vz_{distance}ft"] = vz
    df[f"speed_{distance}ft"] = speed
    df[f"time_{distance}ft"] = t


# ============================================================
# ADDITIONAL TRAJECTORY FEATURES
# ============================================================

for distance in TRAJ_DISTANCES:

    df[f"horizontal_velocity_{distance}ft"] = np.sqrt(
        df[f"vx_{distance}ft"]**2 +
        df[f"vy_{distance}ft"]**2
    )


# ============================================================
# V1 FEATURE LIST
# ============================================================

V1_FEATURES = [
    "pitch_type",

    # Raw pitch physics
    "release_speed",
    "release_spin_rate",
    "spin_axis_sin",
    "spin_axis_cos",

    # Release point
    "release_pos_x",
    "release_pos_y",
    "release_pos_z",
    "release_extension",

    # Initial velocity
    "vx0",
    "vy0",
    "vz0",

    # Acceleration
    "ax",
    "ay",
    "az",

    # Derived physics
    "velocity_mag",
    "acceleration_mag",
    "horizontal_velocity",
    "horizontal_acceleration",
]

for distance in TRAJ_DISTANCES:

    V1_FEATURES += [
        f"x_{distance}ft",
        f"z_{distance}ft",
        f"vx_{distance}ft",
        f"vy_{distance}ft",
        f"vz_{distance}ft",
        f"speed_{distance}ft",
        f"time_{distance}ft",
        f"horizontal_velocity_{distance}ft",
    ]


# ============================================================
# V2: BUILD PITCHER ARSENAL PROFILES (per pitcher x pitch_type x season)
# ============================================================

ARSENAL_GROUP = [
    PITCHER_COL,
    PITCH_TYPE_COL,
    SEASON_COL,
]

ARSENAL_PHYSICS = [
    "release_speed",
    "release_spin_rate",
    "spin_axis_sin",
    "spin_axis_cos",
    "release_pos_x",
    "release_pos_y",
    "release_pos_z",
    "release_extension",
    "vx0",
    "vy0",
    "vz0",
    "ax",
    "ay",
    "az",
]

for distance in TRAJ_DISTANCES:

    ARSENAL_PHYSICS += [
        f"x_{distance}ft",
        f"z_{distance}ft",
        f"vx_{distance}ft",
        f"vy_{distance}ft",
        f"vz_{distance}ft",
        f"speed_{distance}ft",
    ]


# Average physical profile for every pitcher/pitch type/season
arsenal = (
    df
    .groupby(ARSENAL_GROUP, observed=True)[ARSENAL_PHYSICS]
    .mean()
    .reset_index()
)


# ============================================================
# PITCH USAGE (within a pitcher-season)
# ============================================================

arsenal_counts = (
    df
    .groupby(ARSENAL_GROUP, observed=True)
    .size()
    .rename("pitch_count")
    .reset_index()
)

arsenal = arsenal.merge(
    arsenal_counts,
    on=ARSENAL_GROUP,
    how="left"
)

arsenal["pitch_usage"] = (
    arsenal["pitch_count"]
    /
    arsenal.groupby([PITCHER_COL, SEASON_COL], observed=True)["pitch_count"]
    .transform("sum")
)


# ============================================================
# CIRCULAR SPIN-AXIS DIFFERENCE
# ============================================================

def circular_difference_deg(a, b):
    """
    Smallest signed difference between two angles.
    Returns a value from -180 to +180 degrees.

    Pure numpy-ufunc composition -- works identically on scalars or arrays.
    """

    return np.degrees(
        np.arctan2(
            np.sin(np.radians(a - b)),
            np.cos(np.radians(a - b))
        )
    )


# ============================================================
# PAIRWISE FEATURES (VECTORIZED)
# ============================================================
#
# Every pitch is compared against every OTHER pitch type in that
# pitcher's SAME-SEASON arsenal. Instead of looping row-by-row, we
# fan this out via a merge: each pitch gets one row per (pitch,
# other arsenal pitch type), then every gap/separation feature is
# computed as a single vectorized pass over the whole long table.

PAIR_PHYSICS_COLS = [
    "release_speed",
    "release_spin_rate",
    "spin_axis_sin",
    "spin_axis_cos",
    "release_pos_x",
    "release_pos_y",
    "release_pos_z",
    "release_extension",
    "vx0",
    "vy0",
    "vz0",
    "ax",
    "ay",
    "az",
]

PAIR_TRAJ_COLS = []
for distance in TRAJ_DISTANCES:
    PAIR_TRAJ_COLS += [f"x_{distance}ft", f"z_{distance}ft", f"speed_{distance}ft"]

# Stable row id -- must survive unchanged until the sparse block is
# reattached to df below.
df["_orig_idx"] = np.arange(len(df))

pitch_slim = df[
    [PITCHER_COL, PITCH_TYPE_COL, SEASON_COL, "_orig_idx"]
    + PAIR_PHYSICS_COLS
    + PAIR_TRAJ_COLS
].copy()

ref_rename = {col: f"{col}_ref" for col in PAIR_PHYSICS_COLS + PAIR_TRAJ_COLS}
arsenal_ref = (
    arsenal[[PITCHER_COL, PITCH_TYPE_COL, SEASON_COL] + PAIR_PHYSICS_COLS + PAIR_TRAJ_COLS]
    .rename(columns={PITCH_TYPE_COL: "other_type", **ref_rename})
)

# Fan-out join: one row per (pitch, other arsenal pitch type in same pitcher-season)
pairs = pitch_slim.merge(arsenal_ref, on=[PITCHER_COL, SEASON_COL], how="inner")
pairs = pairs[pairs[PITCH_TYPE_COL] != pairs["other_type"]].copy()

del pitch_slim, arsenal_ref

print(f"Pairwise comparison rows: {len(pairs):,}")


def gap(col):
    return pairs[col].to_numpy() - pairs[f"{col}_ref"].to_numpy()


pair_features = {}

# --------------------------------------------------------
# VELOCITY
# --------------------------------------------------------

pair_features["velocity_gap"] = gap("release_speed")
pair_features["abs_velocity_gap"] = np.abs(pair_features["velocity_gap"])


# --------------------------------------------------------
# SPIN
# --------------------------------------------------------

pair_features["spin_rate_gap"] = gap("release_spin_rate")

pitch_axis = np.degrees(
    np.arctan2(
        pairs["spin_axis_sin"].to_numpy(),
        pairs["spin_axis_cos"].to_numpy(),
    )
)

reference_axis = np.degrees(
    np.arctan2(
        pairs["spin_axis_sin_ref"].to_numpy(),
        pairs["spin_axis_cos_ref"].to_numpy(),
    )
)

spin_axis_gap = circular_difference_deg(pitch_axis, reference_axis)

pair_features["spin_axis_gap"] = spin_axis_gap
pair_features["abs_spin_axis_gap"] = np.abs(spin_axis_gap)


# --------------------------------------------------------
# RELEASE POINT
# --------------------------------------------------------

for col, name in [
    ("release_pos_x", "release_x"),
    ("release_pos_y", "release_y"),
    ("release_pos_z", "release_z"),
    ("release_extension", "extension"),
]:

    g = gap(col)
    pair_features[f"{name}_gap"] = g
    pair_features[f"abs_{name}_gap"] = np.abs(g)


# --------------------------------------------------------
# INITIAL VELOCITY
# --------------------------------------------------------

for col in ["vx0", "vy0", "vz0"]:
    pair_features[f"{col}_gap"] = gap(col)


# --------------------------------------------------------
# ACCELERATION
# --------------------------------------------------------

for col in ["ax", "ay", "az"]:
    pair_features[f"{col}_gap"] = gap(col)


# --------------------------------------------------------
# TRAJECTORY SEPARATION
# --------------------------------------------------------

separations = {}

for distance in TRAJ_DISTANCES:

    dx = pairs[f"x_{distance}ft"].to_numpy() - pairs[f"x_{distance}ft_ref"].to_numpy()
    dz = pairs[f"z_{distance}ft"].to_numpy() - pairs[f"z_{distance}ft_ref"].to_numpy()
    separation = np.sqrt(dx**2 + dz**2)
    separations[distance] = separation

    pair_features[f"trajectory_separation_{distance}ft"] = separation
    pair_features[f"x_separation_{distance}ft"] = np.abs(dx)
    pair_features[f"z_separation_{distance}ft"] = np.abs(dz)
    pair_features[f"speed_gap_{distance}ft"] = (
        pairs[f"speed_{distance}ft"].to_numpy() - pairs[f"speed_{distance}ft_ref"].to_numpy()
    )


# --------------------------------------------------------
# TRAJECTORY DIVERGENCE
# --------------------------------------------------------
# Only these features are gated on all four distances being finite --
# the per-distance separation/gap features above are always computed.

finite_mask = np.ones(len(pairs), dtype=bool)
for distance in TRAJ_DISTANCES:
    finite_mask &= np.isfinite(separations[distance])

sep_stack = np.vstack([separations[d] for d in TRAJ_DISTANCES])

pair_features["trajectory_divergence"] = np.where(
    finite_mask,
    separations[TRAJ_DISTANCES[-1]] - separations[TRAJ_DISTANCES[0]],
    np.nan,
)
pair_features["max_trajectory_separation"] = np.where(
    finite_mask, sep_stack.max(axis=0), np.nan
)
pair_features["min_trajectory_separation"] = np.where(
    finite_mask, sep_stack.min(axis=0), np.nan
)

for i in range(len(TRAJ_DISTANCES) - 1):
    d1 = TRAJ_DISTANCES[i]
    d2 = TRAJ_DISTANCES[i + 1]
    pair_features[f"divergence_{d1}_{d2}ft"] = np.where(
        finite_mask, separations[d2] - separations[d1], np.nan
    )


# ============================================================
# NEAREST / FARTHEST / MEAN ARSENAL FEATURES (30ft)
# ============================================================

sep30 = pd.DataFrame({
    "_orig_idx": pairs["_orig_idx"].to_numpy(),
    "sep30": pair_features["trajectory_separation_30ft"],
})
sep30_finite = sep30[np.isfinite(sep30["sep30"])]

agg30 = (
    sep30_finite
    .groupby("_orig_idx")["sep30"]
    .agg(
        nearest_pitch_distance_30ft="min",
        farthest_pitch_distance_30ft="max",
        mean_pitch_distance_30ft="mean",
    )
)

# Reindex onto every pitch (not just those with a valid comparison) so
# pitches with zero valid comparisons get an explicit NaN row.
agg30 = agg30.reindex(df["_orig_idx"])
agg30.index = df.index


# ============================================================
# SPARSE WIDE PIVOT: {current_type}_vs_{other_type}_{feature}
# ============================================================
#
# ~260 type-pair combinations x ~41 features could mean 10,000+ wide
# columns, but each pitch is only non-null for the few other types in
# its own pitcher-season arsenal (~1-2% density). A dense array at this
# shape would need well over 100GB, so this is built as genuinely
# sparse -- never densified as an intermediate step.

pairs["_prefix"] = pairs[PITCH_TYPE_COL].astype(str) + "_vs_" + pairs["other_type"].astype(str)
prefix_codes, prefix_categories = pd.factorize(pairs["_prefix"], sort=True)
row_codes = pairs["_orig_idx"].to_numpy()
n_rows = len(df)
n_prefixes = len(prefix_categories)

sparse_blocks = []

for feat_name, values in pair_features.items():

    mat = scipy.sparse.coo_matrix(
        (values, (row_codes, prefix_codes)),
        shape=(n_rows, n_prefixes),
    ).tocsr()

    col_names = [f"{p}_{feat_name}" for p in prefix_categories]
    block = pd.DataFrame.sparse.from_spmatrix(mat, columns=col_names)

    # Pitch-type pairs absent from a given pitcher-season's arsenal must
    # read as NaN (no comparison made), not 0 -- explicit for portability
    # across pandas/scipy versions even where the default already matches.
    block = block.astype(pd.SparseDtype("float64", np.nan))

    sparse_blocks.append(block)

v2_sparse = pd.concat(sparse_blocks, axis=1)
v2_sparse.index = df.index

del pairs, sparse_blocks


# ============================================================
# FINAL V2 DATAFRAME
# ============================================================

df = df.drop(columns="_orig_idx")

df_v2 = pd.concat([df, v2_sparse, agg30], axis=1)

del v2_sparse


# ============================================================
# USAGE-WEIGHTED ARSENAL FEATURES (VECTORIZED)
# ============================================================
#
# arsenal is small (thousands of rows), so a dense self-merge is fine.

self_pairs = arsenal.merge(arsenal, on=[PITCHER_COL, SEASON_COL], suffixes=("", "_other"))
self_pairs = self_pairs[self_pairs[PITCH_TYPE_COL] != self_pairs[f"{PITCH_TYPE_COL}_other"]].copy()

d30 = np.sqrt(
    (self_pairs["x_30ft"] - self_pairs["x_30ft_other"])**2 +
    (self_pairs["z_30ft"] - self_pairs["z_30ft_other"])**2
)
d40 = np.sqrt(
    (self_pairs["x_40ft"] - self_pairs["x_40ft_other"])**2 +
    (self_pairs["z_40ft"] - self_pairs["z_40ft_other"])**2
)

self_pairs["_weighted_30"] = self_pairs["pitch_usage_other"] * d30
self_pairs["_weighted_40"] = self_pairs["pitch_usage_other"] * d40

usage_grouped = (
    self_pairs
    .groupby([PITCHER_COL, PITCH_TYPE_COL, SEASON_COL], observed=True)
    .agg(
        weighted_30=("_weighted_30", "sum"),
        weighted_40=("_weighted_40", "sum"),
        total_usage=("pitch_usage_other", "sum"),
    )
    .reset_index()
)

usage_grouped["usage_weighted_distance_30ft"] = np.where(
    usage_grouped["total_usage"] > 0,
    usage_grouped["weighted_30"] / usage_grouped["total_usage"],
    np.nan,
)
usage_grouped["usage_weighted_distance_40ft"] = np.where(
    usage_grouped["total_usage"] > 0,
    usage_grouped["weighted_40"] / usage_grouped["total_usage"],
    np.nan,
)

# Left-join back onto the full arsenal (not just usage_grouped's rows) so
# single-pitch-type pitcher-seasons still get an explicit NaN row.
usage_features = arsenal[[PITCHER_COL, PITCH_TYPE_COL, SEASON_COL]].merge(
    usage_grouped[[
        PITCHER_COL, PITCH_TYPE_COL, SEASON_COL,
        "usage_weighted_distance_30ft", "usage_weighted_distance_40ft",
    ]],
    on=[PITCHER_COL, PITCH_TYPE_COL, SEASON_COL],
    how="left",
)

del self_pairs, usage_grouped

# Merge usage features back onto pitch-level data
df_v2 = df_v2.merge(
    usage_features,
    on=[PITCHER_COL, PITCH_TYPE_COL, SEASON_COL],
    how="left",
)


# ============================================================
# FINAL OUTPUTS
# ============================================================
#
# df          -- V1 dataframe (physics + trajectory features)
# df_v2       -- V1 + V2 (arsenal comparison) features. The wide "_vs_"
#                columns are stored as sparse (pandas SparseDtype) --
#                avoid calling .values/.to_numpy() on the whole frame,
#                which would densify them.
# V1_FEATURES -- V1 model feature list
#
# V2 model features are every column in df_v2 containing "_vs_",
# "nearest_pitch", "farthest_pitch", "mean_pitch_distance", or
# "usage_weighted".

## Stuff+ Scoring: PCA Composite &#8594; 100+ Scale

Stuff+ here is a standardized index, not a trained model: pure physics "outlierness" relative to league average for the same pitch type, with no outcome data involved (Location+/Pitching+ handle situational run value later; see `docs/purpose.md`).

**Method:**
1. Per pitch type, z-score a small set of non-redundant physics features within (pitch_type, season) so "average" always means that season's league average for that pitch type.
2. Let PCA (fit pooled across seasons, per pitch type) pick the weights automatically. PC1 is the dominant "how good is this pitch" axis, sign-anchored so higher `release_speed` loads positively.
3. Score every individual pitch AND aggregate to (pitcher, pitch_type, season). Stuff+ is reported at the pitcher-season level (`purpose.md`'s "aggregated to pitch type"), but every pitch also gets a score on the same scale, since Pitching+/bestPitch+ need pitch-level granularity.
4. Convert to a genuine ratio scale: `exp(k * z)`, renormalized so the league average for that pitch type/season is exactly 100. Makes "101 = 1% better than average" literally true, not an SD-relabeling.

In [ ]:
# ============================================================
# STUFF+ SCORING: CONFIGURATION
# ============================================================

STUFF_FEATURES = [
    "release_speed",
    "release_spin_rate",
    "release_extension",
    "acceleration_mag",
    "horizontal_acceleration",
    "movement_per_reaction_time",
]

# Minimum pitches in a (pitcher, pitch_type, season) group for its average to
# set the league reference distribution / be reported as "reliable."
MIN_PITCHES_FOR_SCORE = 20

# Controls how many points one SD of stuff is worth on the final scale
# (0.10 -> roughly +/-10 points per SD, in line with typical "+" stat spreads).
STUFF_SCALE_K = 0.10


# ============================================================
# "REACTION x MOVEMENT": movement packed into the available reaction window
# ============================================================
# Higher = more break, delivered in less time -- purpose.md's explicit
# "Reaction x Movement" concept. Testing found this the single highest-
# loading PC1 feature across every pitch type.

df["movement_per_reaction_time"] = df["horizontal_acceleration"] / df["time_30ft"]

stuff_df = df[np.isfinite(df["movement_per_reaction_time"])].copy()


# ============================================================
# PER-PITCH COMPOSITE (PCA, sign-anchored so higher = better)
# ============================================================

stuff_df["pitch_composite"] = np.nan
pca_loadings = {}

for ptype, type_group in stuff_df.groupby(PITCH_TYPE_COL, observed=True):

    if len(type_group) < 5000:
        continue

    # z-score within (pitch_type, season) so "average" means that season's average
    season_mu = type_group.groupby(SEASON_COL)[STUFF_FEATURES].transform("mean")
    season_sigma = type_group.groupby(SEASON_COL)[STUFF_FEATURES].transform("std")
    Z = (type_group[STUFF_FEATURES] - season_mu) / season_sigma

    # PCA weights fit pooled across seasons -- more stable than fitting per season
    pca = PCA(n_components=len(STUFF_FEATURES))
    pca.fit(Z.to_numpy())

    loadings = pca.components_[0]
    if loadings[STUFF_FEATURES.index("release_speed")] < 0:
        loadings = -loadings

    pca_loadings[ptype] = loadings
    stuff_df.loc[type_group.index, "pitch_composite"] = Z.to_numpy() @ loadings

pca_loadings_df = pd.DataFrame(pca_loadings, index=STUFF_FEATURES).T
pca_loadings_df

In [9]:
# ============================================================
# AGGREGATE TO (pitcher, pitch_type, season)
# ============================================================
# Stuff+ is reported at this level (purpose.md: "aggregated to pitch type"),
# matching how real "+" stats (wRC+, ERA-) work.

pitcher_agg = (
    stuff_df
    .dropna(subset=["pitch_composite"])
    .groupby([PITCHER_COL, PITCH_TYPE_COL, SEASON_COL, "player_name"], observed=True)["pitch_composite"]
    .agg(mean_composite="mean", n_pitches="count")
    .reset_index()
)


# ============================================================
# CALIBRATION: league reference distribution, per (pitch_type, season)
# ============================================================
# Only "reliable" (large enough sample) pitcher-seasons set the reference --
# a tiny sample's own mean is noisy and would distort the league average/SD
# used to calibrate the scale for everyone.

reliable = pitcher_agg[pitcher_agg["n_pitches"] >= MIN_PITCHES_FOR_SCORE].copy()

calibration = (
    reliable
    .groupby([PITCH_TYPE_COL, SEASON_COL])["mean_composite"]
    .agg(agg_mu="mean", agg_sigma="std")
    .reset_index()
)
reliable = reliable.merge(calibration, on=[PITCH_TYPE_COL, SEASON_COL], how="left")
reliable["agg_z"] = (reliable["mean_composite"] - reliable["agg_mu"]) / reliable["agg_sigma"]
reliable["raw_ratio"] = np.exp(STUFF_SCALE_K * reliable["agg_z"])

# raw_ratio_mean anchors the scale to exactly 100 -- computed from the SAME
# reliable pitcher-seasons only, so unreliable small samples can't skew it.
raw_ratio_mean = (
    reliable
    .groupby([PITCH_TYPE_COL, SEASON_COL])["raw_ratio"]
    .mean()
    .rename("raw_ratio_mean")
    .reset_index()
)
calibration = calibration.merge(raw_ratio_mean, on=[PITCH_TYPE_COL, SEASON_COL], how="left")

del reliable


# ============================================================
# APPLY THE SAME CALIBRATION TO BOTH LEVELS -- SHARED SCALE, SHARED "100"
# ============================================================
# Pitch-level and pitcher-season scores share the same (agg_mu, agg_sigma,
# raw_ratio_mean) constants, so averaging a pitcher's pitch-level scores for
# a season lands very close to (not exactly -- exp is nonlinear, Jensen's
# inequality gives a small systematic +0.2-0.3 point gap on average, up to
# a few points in rare small-sample cases -- but not exactly equal to) their
# aggregate season score.

pitcher_agg = pitcher_agg.merge(calibration, on=[PITCH_TYPE_COL, SEASON_COL], how="left")
pitcher_agg["stuff_plus"] = 100 * np.exp(
    STUFF_SCALE_K * (pitcher_agg["mean_composite"] - pitcher_agg["agg_mu"]) / pitcher_agg["agg_sigma"]
) / pitcher_agg["raw_ratio_mean"]
pitcher_agg["reliable"] = pitcher_agg["n_pitches"] >= MIN_PITCHES_FOR_SCORE

stuff_df = stuff_df.merge(calibration, on=[PITCH_TYPE_COL, SEASON_COL], how="left")
stuff_df["pitch_stuff_plus"] = 100 * np.exp(
    STUFF_SCALE_K * (stuff_df["pitch_composite"] - stuff_df["agg_mu"]) / stuff_df["agg_sigma"]
) / stuff_df["raw_ratio_mean"]


# ============================================================
# OUTPUTS
# ============================================================
#
# Pitch-level Stuff+ (every individual pitch -- feed this into Pitching+/
# bestPitch+, which need pitch-by-pitch granularity to combine with Location+):
#
#     stuff_df["pitch_stuff_plus"]
#
# Pitcher x pitch_type x season Stuff+ (headline reporting figure, filtered
# to reliable -- i.e. large enough sample -- pitcher-seasons):
#
#     pitcher_stuff_plus

pitcher_stuff_plus = (
    pitcher_agg[pitcher_agg["reliable"]]
    .sort_values("stuff_plus", ascending=False)
    .reset_index(drop=True)
)

pitcher_stuff_plus.head(15)

,pitcher,pitch_type,game_year,player_name,mean_composite,n_pitches,agg_mu,agg_sigma,raw_ratio_mean,stuff_plus,reliable
0,666808,FC,2023,"Doval, Camilo",5.545878,381,-0.195919,1.227604,1.005076,158.829411,True
1,661403,FC,2021,"Clase, Emmanuel",5.067723,722,-0.128132,1.204704,1.005139,153.138384,True
2,612434,FF,2022,"Castro, Miguel",6.126630,24,0.016096,1.438261,1.005026,152.172122,True
3,666808,FC,2024,"Doval, Camilo",4.749027,378,-0.176218,1.209361,1.005053,149.513947,True
4,661403,FC,2024,"Clase, Emmanuel",4.720034,765,-0.176218,1.209361,1.005053,149.155940,True
5,666808,FC,2022,"Doval, Camilo",4.867182,336,-0.118163,1.234258,1.005136,149.002227,True
6,666808,FC,2021,"Doval, Camilo",4.703472,180,-0.128132,1.204704,1.005139,148.577433,True
7,690829,FF,2023,"Joyce, Ben",5.527812,156,-0.082642,1.443684,1.005029,146.756482,True
8,661395,FS,2025,"Duran, Jhoan",5.478680,428,-0.194454,1.462118,1.005090,146.657701,True
9,661403,FC,2022,"Clase, Emmanuel",4.662004,549,-0.118163,1.234258,1.005136,146.545752,True


In [14]:
# ============================================================
# TOP 10 PER PITCH TYPE (ONE SEASON PER PITCHER -- THEIR BEST)
# ============================================================

best_season_idx = (
    pitcher_stuff_plus
    .groupby([PITCHER_COL, PITCH_TYPE_COL])["stuff_plus"]
    .idxmax()
)

best_season = pitcher_stuff_plus.loc[best_season_idx]

top10_by_type = (
    best_season
    .sort_values([PITCH_TYPE_COL, "stuff_plus"], ascending=[True, False])
    .groupby(PITCH_TYPE_COL)
    .head(10)
    [[PITCH_TYPE_COL, "player_name", SEASON_COL, "n_pitches", "stuff_plus"]]
    .reset_index(drop=True)
)

top10_by_type["rank"] = top10_by_type.groupby(PITCH_TYPE_COL).cumcount() + 1
top10_by_type = top10_by_type[[PITCH_TYPE_COL, "rank", "player_name", SEASON_COL, "n_pitches", "stuff_plus"]]

display(top10_by_type[top10_by_type['pitch_type'] == 'FS'])

,pitch_type,rank,player_name,game_year,n_pitches,stuff_plus
40,FS,1,"Duran, Jhoan",2025,428,146.657701
41,FS,2,"Soriano, José",2023,99,142.008806
42,FS,3,"Skenes, Paul",2024,603,133.938170
43,FS,4,"Familia, Jeurys",2021,30,133.351841
44,FS,5,"Cano, Yennier",2022,38,132.903642
45,FS,6,"Marte, Yunior",2024,30,130.830042
46,FS,7,"Boyle, Joe",2025,152,126.590818
47,FS,8,"Walker, Taijuan",2021,363,122.513917
48,FS,9,"Hoffman, Jeff",2024,119,120.034719
49,FS,10,"Bazardo, Eduard",2024,32,117.810839
